# GAT-v4 | Phase C1 — CPTAC Acquisition & Alignment

Builds `cptac_clinical_clean.csv` and `cptac_sample_alignment.csv` from your
three raw CPTAC sources — mirrors TCGA Phase 1's alignment logic, adapted to
CPTAC's file layout.

**Cohort confirmed clean:** all 236 patients are `Renal cell carcinoma, NOS`
(ICD-O `8312/3`) — no clear-cell filtering needed.

**Sources:**
| Data | File | Used for |
|---|---|---|
| GDC clinical | `clinical.tsv` | stage, grade |
| cBioPortal clinical | `data_clinical_patient.txt` | age, OS_MONTHS, OS_STATUS |
| GDC sample sheet | `gdc_sample_sheet.*.tsv` | Case ID -> tumor sample selection |

**Multi-sample rule:** 40 patients have >1 tumor RNA sample. We keep
`Tumor Descriptor == 'Primary'` samples only, then take the lowest Sample ID
if a patient still has more than one — one tumor sample per patient, same
convention as TCGA.

---
| Cell | What |
|------|------|
| 1 | Imports & paths (EDIT paths) |
| 2 | Load & filter GDC sample sheet -> one tumor sample per patient |
| 3 | Load GDC clinical.tsv -> stage/grade |
| 4 | Load cBioPortal clinical_patient.txt -> age/OS |
| 5 | Merge -> cptac_clinical_clean.csv |
| 6 | Build cptac_sample_alignment.csv |
| 7 | Save + audit |


---
## Cell 1 — Imports & Paths

**Edit these paths to match your machine.**


In [2]:
import sys, json, yaml, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# ── TCGA project root (for logger/config reuse — optional) ───────────────────
PROJECT_ROOT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")
KIRC_ROOT    = PROJECT_ROOT / "KIRC"

# ── CPTAC raw data paths — EDIT THESE ─────────────────────────────────────────
CPTAC_ROOT        = Path(r"C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data")
CPTAC_RNA_ROOT    = CPTAC_ROOT / "RNA"
CPTAC_GENE_EXPR_DIR = CPTAC_RNA_ROOT / "gene expression"          # per-patient folders
CPTAC_CLINICAL_TSV  = CPTAC_RNA_ROOT / "clinical.tsv"
CPTAC_SAMPLE_SHEET  = CPTAC_RNA_ROOT / "gdc_sample_sheet.2026-07-12.tsv"
CPTAC_CBIOPORTAL_DIR = CPTAC_ROOT / "rcc_cptac_gdc"
CPTAC_CLINICAL_PATIENT_TXT = CPTAC_CBIOPORTAL_DIR / "data_clinical_patient.txt"

# ── Output: CPTAC processed dir (this is what Phase 9 reads from) ────────────
CPTAC_DATA_PROC = KIRC_ROOT / "data" / "cptac_processed"
CPTAC_DATA_PROC.mkdir(parents=True, exist_ok=True)
AUDIT_DIR = KIRC_ROOT / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

for p, name in [(CPTAC_GENE_EXPR_DIR, 'RNA gene expression dir'),
                (CPTAC_CLINICAL_TSV, 'GDC clinical.tsv'),
                (CPTAC_SAMPLE_SHEET, 'GDC sample sheet'),
                (CPTAC_CLINICAL_PATIENT_TXT, 'cBioPortal clinical_patient.txt')]:
    print(f"  {'OK  ' if p.exists() else 'MISSING'}  {name:<28} -> {p}")

print(f"\nOutput dir: {CPTAC_DATA_PROC}")


  OK    RNA gene expression dir      -> C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data\RNA\gene expression
  OK    GDC clinical.tsv             -> C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data\RNA\clinical.tsv
  OK    GDC sample sheet             -> C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data\RNA\gdc_sample_sheet.2026-07-12.tsv
  OK    cBioPortal clinical_patient.txt -> C:\Users\jiyad\Desktop\PROJECT\CPTAC\raw_data\rcc_cptac_gdc\data_clinical_patient.txt

Output dir: C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\cptac_processed


---
## Cell 2 — Load & Filter GDC Sample Sheet -> One Tumor Sample Per Patient

Handles the comma-separated multi-value rows (some rows list 2 aliquots on
one line) and the 40 patients with >1 tumor sample.


In [3]:
sheet = pd.read_csv(CPTAC_SAMPLE_SHEET, sep='\t')
print(f"Sample sheet rows      : {len(sheet)}")
print(f"Columns                : {list(sheet.columns)}")

# ── Explode comma-separated multi-value rows (Case ID / Sample ID / Tissue Type / Tumor Descriptor) ──
multi_cols = ['Case ID', 'Sample ID', 'Tissue Type', 'Tumor Descriptor']
rows = []
for _, row in sheet.iterrows():
    split_vals = {c: [v.strip() for v in str(row[c]).split(',')] for c in multi_cols}
    n_split = max(len(v) for v in split_vals.values())
    for i in range(n_split):
        new_row = row.to_dict()
        for c in multi_cols:
            vals = split_vals[c]
            new_row[c] = vals[i] if i < len(vals) else vals[-1]
        rows.append(new_row)
sheet_exp = pd.DataFrame(rows)
print(f"After exploding multi-value rows: {len(sheet_exp)}")

# ── Keep tumor samples only ────────────────────────────────────────────────────
tumor = sheet_exp[sheet_exp['Tissue Type'].str.contains('Tumor', na=False)].copy()
print(f"Tumor rows              : {len(tumor)}")
print(f"Unique tumor cases       : {tumor['Case ID'].nunique()}")

# ── Rule: prefer 'Primary' descriptor, then lowest Sample ID if still >1 ────────
tumor['is_primary'] = tumor['Tumor Descriptor'].str.strip().eq('Primary')
tumor_sorted = tumor.sort_values(['Case ID', 'is_primary', 'Sample ID'],
                                  ascending=[True, False, True])
tumor_dedup = tumor_sorted.drop_duplicates('Case ID', keep='first')

print(f"\nAfter one-tumor-per-patient rule: {len(tumor_dedup)} patients")
n_non_primary = (~tumor_dedup['is_primary']).sum()
if n_non_primary > 0:
    print(f"  NOTE: {n_non_primary} patients had no 'Primary' sample; used next available tumor sample.")

# case_id -> chosen tumor File Name / Sample ID
case_to_file = dict(zip(tumor_dedup['Case ID'], tumor_dedup['File Name']))
case_to_sample_id = dict(zip(tumor_dedup['Case ID'], tumor_dedup['Sample ID']))
print(f"\ncase_to_file entries: {len(case_to_file)}")
print(tumor_dedup[['Case ID', 'Sample ID', 'Tumor Descriptor', 'File Name']].head(5).to_string(index=False))


Sample sheet rows      : 466
Columns                : ['File ID', 'File Name', 'Data Category', 'Data Type', 'Project ID', 'Case ID', 'Sample ID', 'Tissue Type', 'Tumor Descriptor', 'Specimen Type', 'Preservation Method']
After exploding multi-value rows: 495
Tumor rows              : 342
Unique tumor cases       : 223

After one-tumor-per-patient rule: 223 patients

case_to_file entries: 223
  Case ID    Sample ID Tumor Descriptor                                                                   File Name
C3L-00004 C3L-00004-01          Primary 34226159-5237-455f-ac9b-2713eba4d87e.rna_seq.augmented_star_gene_counts.tsv
C3L-00010 C3L-00010-01          Primary d7fa7d14-7477-4229-83da-a6af5c0ba63f.rna_seq.augmented_star_gene_counts.tsv
C3L-00011 C3L-00011-01          Primary 84c6e8e2-714e-4c11-992d-0e36bb01d83a.rna_seq.augmented_star_gene_counts.tsv
C3L-00026 C3L-00026-01          Primary 59ba3c66-521b-4224-ace6-62e7a2c89a65.rna_seq.augmented_star_gene_counts.tsv
C3L-00079 C3L-00079-01  

---
## Cell 3 — Load GDC `clinical.tsv` -> Stage & Grade

Uses the exact `encode_stage()`/`encode_grade()` conventions from TCGA Phase 1
so downstream numeric codes (1-4) are identical across cohorts.


In [4]:
clin_gdc = pd.read_csv(CPTAC_CLINICAL_TSV, sep='\t')
print(f"GDC clinical rows : {len(clin_gdc)}")

COL_CASE  = 'cases.submitter_id'   # e.g. C3L-00011 — matches Case ID in sample sheet
COL_STAGE = 'diagnoses.ajcc_pathologic_stage'
COL_GRADE = 'diagnoses.tumor_grade'

for c in [COL_CASE, COL_STAGE, COL_GRADE]:
    print(f"  {'OK  ' if c in clin_gdc.columns else 'MISSING'}  {c}")

def encode_stage(val):
    if pd.isna(val):
        return np.nan
    v = str(val).upper().strip()
    if v.startswith("STAGE IV"):   return 4
    elif v.startswith("STAGE III"): return 3
    elif v.startswith("STAGE II"):  return 2
    elif v.startswith("STAGE I"):   return 1
    return np.nan

def encode_grade(val):
    if pd.isna(val):
        return np.nan
    v = str(val).upper().strip()
    for i in [1, 2, 3, 4]:
        if f'G{i}' in v or f'GRADE {i}' in v or str(i) in v:
            return i
    return np.nan

stage_grade = clin_gdc[[COL_CASE, COL_STAGE, COL_GRADE]].drop_duplicates(COL_CASE).copy()
stage_grade['stage'] = stage_grade[COL_STAGE].apply(encode_stage)
stage_grade['grade'] = stage_grade[COL_GRADE].apply(encode_grade)
stage_grade = stage_grade.rename(columns={COL_CASE: 'patient_id'})[['patient_id', 'stage', 'grade']]

print(f"\nPatients with stage : {stage_grade['stage'].notna().sum()} / {len(stage_grade)}")
print(f"Patients with grade : {stage_grade['grade'].notna().sum()} / {len(stage_grade)}")
print(stage_grade['stage'].value_counts(dropna=False).sort_index())
print(stage_grade['grade'].value_counts(dropna=False).sort_index())


GDC clinical rows : 236
  OK    cases.submitter_id
  OK    diagnoses.ajcc_pathologic_stage
  OK    diagnoses.tumor_grade

Patients with stage : 206 / 223
Patients with grade : 223 / 223
stage
1.0    104
2.0     22
3.0     57
4.0     23
NaN     17
Name: count, dtype: int64
grade
1     19
2    114
3     68
4     22
Name: count, dtype: int64


---
## Cell 4 — Load cBioPortal `data_clinical_patient.txt` -> Age & Survival

cBioPortal clinical-data files have a 4-line header block (`#`-prefixed) before
the real header row — same shape as your TCGA cBioPortal downloads.


In [5]:
clin_cbio = pd.read_csv(CPTAC_CLINICAL_PATIENT_TXT, sep='\t', comment='#')
print(f"cBioPortal clinical_patient rows : {len(clin_cbio)}")
print(f"Columns : {list(clin_cbio.columns)}")

def encode_event(val):
    if pd.isna(val):
        return np.nan
    v = str(val).upper()
    if '1' in v or 'DECEASED' in v or 'DEAD' in v: return 1
    if '0' in v or 'LIVING'   in v or 'ALIVE' in v: return 0
    return np.nan

survival = clin_cbio[['PATIENT_ID', 'OS_STATUS', 'OS_MONTHS', 'AGE']].copy()
survival['event'] = survival['OS_STATUS'].apply(encode_event)
survival['os_months'] = pd.to_numeric(survival['OS_MONTHS'], errors='coerce')
survival['age'] = pd.to_numeric(survival['AGE'], errors='coerce')
survival = survival.rename(columns={'PATIENT_ID': 'patient_id'})[
    ['patient_id', 'event', 'os_months', 'age']]

print(f"\nPatients with event/os_months : {survival[['event','os_months']].notna().all(axis=1).sum()} / {len(survival)}")
print(f"Patients with age              : {survival['age'].notna().sum()} / {len(survival)}")
print(f"Events (deceased)              : {int(survival['event'].sum(skipna=True))}")
print(f"Censored (alive)               : {int((survival['event']==0).sum())}")

# ── os_months fallback: if cBioPortal's OS_MONTHS is missing, derive it from GDC
#    clinical.tsv day-count fields — days_to_death for deceased patients,
#    days_to_last_follow_up for censored (alive) patients. Using the wrong field
#    for a given event status would systematically understate survival time for
#    the deceased, so the choice is event-aware, not a single blanket column.
#    Days -> months uses days/(365.25/12), the same year-basis as the age
#    fallback above, so all derived time units in this table are consistent. ──
DAYS_PER_MONTH = 365.25 / 12

n_missing_os = survival['os_months'].isna().sum()
if n_missing_os > 0:
    print(f"\n{n_missing_os} patients missing OS_MONTHS from cBioPortal — "
          f"filling from GDC clinical.tsv day-count fields...")

    days_cols = ['cases.submitter_id', 'demographic.days_to_death',
                 'diagnoses.days_to_last_follow_up']
    missing_days_cols = [c for c in days_cols if c not in clin_gdc.columns]
    if missing_days_cols:
        print(f"  WARNING: columns not found in clinical.tsv: {missing_days_cols} — "
              f"skipping os_months fallback for those fields.")

    os_fallback = clin_gdc[[c for c in days_cols if c in clin_gdc.columns]].copy()
    os_fallback = os_fallback.rename(columns={'cases.submitter_id': 'patient_id'})
    for c in ['demographic.days_to_death', 'diagnoses.days_to_last_follow_up']:
        if c in os_fallback.columns:
            os_fallback[c] = pd.to_numeric(os_fallback[c], errors='coerce')
    os_fallback = os_fallback.drop_duplicates('patient_id')

    n_before_os_merge = len(survival)
    survival = survival.merge(os_fallback, on='patient_id', how='left')
    assert len(survival) == n_before_os_merge, \
        f"Row count changed after os_months-fallback merge ({n_before_os_merge} -> {len(survival)}) — " \
        f"duplicate patient_id values slipped through, investigate before proceeding."

    days_to_death = survival.get('demographic.days_to_death')
    days_to_fu    = survival.get('diagnoses.days_to_last_follow_up')
    is_deceased   = survival['event'] == 1

    os_days_fallback = pd.Series(np.nan, index=survival.index)
    if days_to_death is not None:
        os_days_fallback = os_days_fallback.where(~is_deceased, days_to_death)
    if days_to_fu is not None:
        os_days_fallback = os_days_fallback.where(is_deceased, days_to_fu)

    os_months_fallback = os_days_fallback / DAYS_PER_MONTH

    n_filled = (survival['os_months'].isna() & os_months_fallback.notna()).sum()
    survival['os_months'] = survival['os_months'].fillna(os_months_fallback)
    survival = survival.drop(columns=[c for c in
        ['demographic.days_to_death', 'diagnoses.days_to_last_follow_up'] if c in survival.columns])

    print(f"  Filled {n_filled} / {n_missing_os} missing os_months from GDC day-count fields "
          f"(deceased <- days_to_death, censored <- days_to_last_follow_up).")
    print(f"  Still missing after fallback: {survival['os_months'].isna().sum()}")

if survival['age'].isna().any():
    print(f"\n{survival['age'].isna().sum()} patients missing age from cBioPortal — "
          f"checking diagnoses.age_at_diagnosis (days) fallback...")
    age_fallback = clin_gdc[['cases.submitter_id', 'diagnoses.age_at_diagnosis']].copy()
    age_fallback['age_fallback'] = pd.to_numeric(
        age_fallback['diagnoses.age_at_diagnosis'], errors='coerce') / 365.25
    age_fallback = age_fallback.rename(columns={'cases.submitter_id': 'patient_id'})
    # IMPORTANT: clin_gdc can have >1 row per patient (diagnosis/treatment-level table) —
    # dedupe BEFORE merging, same guard as the os_months fallback above, or patients with
    # multiple GDC rows get duplicated in `survival`.
    age_fallback = age_fallback.dropna(subset=['age_fallback']).drop_duplicates('patient_id')
    n_before_merge = len(survival)
    survival = survival.merge(age_fallback[['patient_id', 'age_fallback']], on='patient_id', how='left')
    assert len(survival) == n_before_merge, \
        f"Row count changed after age-fallback merge ({n_before_merge} -> {len(survival)}) — " \
        f"duplicate patient_id values slipped through, investigate before proceeding."
    survival['age'] = survival['age'].fillna(survival['age_fallback'])
    survival = survival.drop(columns=['age_fallback'])
    print(f"After fallback, patients with age : {survival['age'].notna().sum()} / {len(survival)}")


cBioPortal clinical_patient rows : 261
Columns : ['PATIENT_ID', 'CAUSE_OF_DEATH', 'DAYS_TO_BIRTH', 'DAYS_TO_DEATH', 'ETHNICITY', 'PROJECT_ID', 'PRIMARY_DIAGNOSIS', 'PRIMARY_SITE_PATIENT', 'RACE', 'SEX', 'VITAL_STATUS', 'OS_STATUS', 'OS_MONTHS', 'AGE']

Patients with event/os_months : 36 / 261
Patients with age              : 244 / 261
Events (deceased)              : 36
Censored (alive)               : 208

225 patients missing OS_MONTHS from cBioPortal — filling from GDC clinical.tsv day-count fields...
  Filled 173 / 225 missing os_months from GDC day-count fields (deceased <- days_to_death, censored <- days_to_last_follow_up).
  Still missing after fallback: 52

17 patients missing age from cBioPortal — checking diagnoses.age_at_diagnosis (days) fallback...
After fallback, patients with age : 244 / 261


---
## Cell 5 — Merge -> `cptac_clinical_clean.csv`

Same required-field rule as TCGA Phase 1: keep patients with valid
`event`/`os_months` (and `os_months > 0`); age/stage/grade are kept if present
but not required (Phase 9's per-fold statistics impute missing values from
the TCGA train-fold median, same as internal TCGA CV).


In [6]:
clin_merged = survival.merge(stage_grade, on='patient_id', how='left')

clin_clean = clin_merged[
    clin_merged[['event', 'os_months']].notna().all(axis=1) &
    (clin_merged['os_months'] > 0)
].drop_duplicates('patient_id').copy()

print(f"Patients total (survival source)  : {len(survival)}")
print(f"Complete records (event+os_months): {len(clin_clean)}")
print(f"Events (deceased)                 : {int(clin_clean['event'].sum())}")
print(f"Censored (alive)                  : {int((clin_clean['event']==0).sum())}")
print(f"With age                          : {clin_clean['age'].notna().sum()}")
print(f"With stage                        : {clin_clean['stage'].notna().sum()}")
print(f"With grade                        : {clin_clean['grade'].notna().sum()}")

clin_out_path = CPTAC_DATA_PROC / 'cptac_clinical_clean.csv'
clin_clean.to_csv(clin_out_path, index=False)
print(f"\nSaved -> {clin_out_path}")


Patients total (survival source)  : 261
Complete records (event+os_months): 202
Events (deceased)                 : 35
Censored (alive)                  : 167
With age                          : 202
With stage                        : 199
With grade                        : 199

Saved -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\cptac_processed\cptac_clinical_clean.csv


---
## Cell 6 — Build `cptac_sample_alignment.csv`

Only patients present in both the clean clinical table (Cell 5) and the
chosen-tumor-sample map (Cell 2) are included — `include=True` for all rows,
since filtering already happened above.


In [7]:
align_rows = []
for pat in clin_clean['patient_id']:
    if pat not in case_to_sample_id:
        continue
    align_rows.append({
        'sample_id': case_to_sample_id[pat],
        'patient_id': pat,
        'tissue_type': 'tumor',
        'include': True,
        'rna_file_name': case_to_file[pat],
    })
align_df = pd.DataFrame(align_rows)

print(f"Clinical patients          : {len(clin_clean)}")
print(f"With a mapped tumor sample : {len(align_df)}")
missing = set(clin_clean['patient_id']) - set(align_df['patient_id'])
if missing:
    print(f"  {len(missing)} clinical patients have no RNA sample — dropped from alignment: {list(missing)[:5]}")

align_out_path = CPTAC_DATA_PROC / 'cptac_sample_alignment.csv'
align_df.to_csv(align_out_path, index=False)
print(f"\nSaved -> {align_out_path}")
print(align_df.head())


Clinical patients          : 202
With a mapped tumor sample : 199
  3 clinical patients have no RNA sample — dropped from alignment: ['C3N-02818', 'C3N-02440', 'C3N-01653']

Saved -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\cptac_processed\cptac_sample_alignment.csv
      sample_id patient_id tissue_type  include  \
0  C3L-00004-01  C3L-00004       tumor     True   
1  C3L-00010-01  C3L-00010       tumor     True   
2  C3L-00011-01  C3L-00011       tumor     True   
3  C3L-00026-01  C3L-00026       tumor     True   
4  C3L-00079-01  C3L-00079       tumor     True   

                                       rna_file_name  
0  34226159-5237-455f-ac9b-2713eba4d87e.rna_seq.a...  
1  d7fa7d14-7477-4229-83da-a6af5c0ba63f.rna_seq.a...  
2  84c6e8e2-714e-4c11-992d-0e36bb01d83a.rna_seq.a...  
3  59ba3c66-521b-4224-ace6-62e7a2c89a65.rna_seq.a...  
4  24314fd5-f34b-417f-862b-98988c2a92b1.rna_seq.a...  


---
## Cell 7 — Audit YAML


In [8]:
audit = {
    'phase': 'Phase C1 — CPTAC Acquisition & Alignment',
    'timestamp': datetime.now().isoformat(),
    'cohort_histology_check': {
        'primary_diagnosis': 'Renal cell carcinoma, NOS (100%)',
        'morphology_icdo': '8312/3 (100%)',
        'filtering_needed': False,
    },
    'n_patients_clinical_source': int(len(survival)),
    'n_patients_clean': int(len(clin_clean)),
    'n_patients_aligned_to_rna': int(len(align_df)),
    'n_events': int(clin_clean['event'].sum()),
    'n_censored': int((clin_clean['event'] == 0).sum()),
    'os_months_filled_from_gdc_days': int(n_filled) if 'n_filled' in dir() else 0,
    'n_with_age': int(clin_clean['age'].notna().sum()),
    'n_with_stage': int(clin_clean['stage'].notna().sum()),
    'n_with_grade': int(clin_clean['grade'].notna().sum()),
    'multi_tumor_sample_patients': int(n_non_primary) if 'n_non_primary' in dir() else None,
    'outputs': {
        'clinical_clean': str(clin_out_path),
        'sample_alignment': str(align_out_path),
    },
    'status': 'COMPLETE',
}
audit_path = AUDIT_DIR / 'phaseC1_cptac_acquisition_alignment.yaml'
with open(audit_path, 'w', encoding='utf-8') as f:
    yaml.dump(audit, f, default_flow_style=False, sort_keys=False)
print(f"Audit YAML -> {audit_path}")
print("\nPhase C1 complete.")


Audit YAML -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\audit\phaseC1_cptac_acquisition_alignment.yaml

Phase C1 complete.
